# Pipeline Job — Taco Bueno Pricing (Orquestador)

Ejecuta el pipeline Medallion completo en orden estricto y verifica que las tablas finales estén disponibles y sin checks FALLIDOS antes de terminar:

1. **03_Pipeline_Silver** — Bronze → Silver (dedup, tipos, cuarentena, reconciliación, DQ).
2. **04_Pipeline_Gold** — Silver → Gold (6 tablas analíticas + DQ de reconciliación cruzada).

Ambos notebooks son **idempotentes** (CTAS + delete/insert por capa): un rerun nunca duplica filas ni double-counting de revenue. Si algún assert de DQ falla, el Job termina en ERROR y el resultado queda visible en el historial del Job.

**Diseño:** una sola tarea secuencial (`%run`) — la dependencia 03 → 04 está garantizada por el orden de ejecución y los asserts internos (04 no puede correr si la Silver falló).

**Guard de concurrencia (Fase 3):** antes de ejecutar, la corrida toma un lock en `taco_bueno_pipeline_lock` para que dos ejecuciones (Jobs o interactivas) no escriban las mismas tablas a la vez — la causa más probable de los conflictos de escritura observados el 28-oct. Un lock huérfano (corrida que falló antes de liberar) expira solo tras 30 min.

In [0]:
# Guard de concurrencia (Fase 3): evita que dos ejecuciones (Jobs o interactivas)
# escriban las mismas tablas a la vez. Un lock huérfano (corrida que falló antes de
# liberar) expira solo tras LOCK_TTL_MIN minutos.
import uuid

LOCK_TTL_MIN = 30  # el pipeline completo tarda ~5-10 min; margen generoso

dbutils.widgets.text("pipeline_cadence", "manual")
job_cadence = dbutils.widgets.get("pipeline_cadence")

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.default.taco_bueno_pipeline_lock (
  lock_token STRING, acquired_at TIMESTAMP, released_at TIMESTAMP, run_source STRING
) USING DELTA
""")
# Expirar locks huérfanos (corridas que terminaron sin liberar)
spark.sql(f"""
DELETE FROM workspace.default.taco_bueno_pipeline_lock
WHERE acquired_at < current_timestamp() - INTERVAL {LOCK_TTL_MIN} MINUTES
""")
active = spark.sql("""
SELECT COUNT(*) AS n FROM workspace.default.taco_bueno_pipeline_lock
WHERE released_at IS NULL
""").collect()[0]['n']
assert active == 0, (
    f"ERROR: hay {active} lock(s) activo(s) del pipeline (taco_bueno_pipeline_lock). "
    "Otra ejecución está en curso o quedó un lock sin liberar (expira solo tras "
    f"{LOCK_TTL_MIN} min). No se ejecuta el pipeline para evitar escrituras concurrentes."
)
run_token = str(uuid.uuid4())
spark.sql(f"""
INSERT INTO workspace.default.taco_bueno_pipeline_lock
VALUES ('{run_token}', current_timestamp(), NULL, '05_Pipeline_Job:{job_cadence}')
""")
print(f"🔒 Lock adquirido ({run_token[:8]}…) — cadencia: {job_cadence}")

In [0]:
%run "/Workspace/Taco Bueno Challenge/Datalithics - Taco Bueno Challenge/Notebooks/03_Pipeline_Silver"

In [0]:
%run "/Workspace/Taco Bueno Challenge/Datalithics - Taco Bueno Challenge/Notebooks/04_Pipeline_Gold"

In [0]:
# Validación final del pipeline: tablas Gold disponibles + DQ sin FAIL
tablas = spark.sql("""
SELECT table_name FROM system.information_schema.tables
WHERE table_schema = 'default' AND table_name LIKE 'taco_bueno_gold%'
ORDER BY table_name
""").collect()
n_gold = len(tablas)
print(f"Tablas Gold disponibles: {n_gold}")
for t in tablas:
    print(f"  • workspace.default.{t['table_name']}")

assert n_gold == 6, f"ERROR: se esperaban 6 tablas Gold, hay {n_gold}"

dq_final = spark.sql("""
SELECT layer, COUNT(*) AS checks,
       SUM(CASE WHEN status = 'FAIL' THEN 1 ELSE 0 END) AS fails
FROM workspace.default.taco_bueno_dq_results
GROUP BY layer
""").collect()
for r in dq_final:
    print(f"DQ [{r['layer']}]: {r['checks']} checks, {r['fails']} FAIL")
    assert r['fails'] == 0, f"ERROR: DQ capa {r['layer']} con checks FAIL"

print("\n✅ PIPELINE COMPLETO: Silver + Gold verificadas, DQ sin fallos. Job OK.")

## Paso 3 — Snapshot de KPIs (historial de corridas)

Cada corrida del Job **agrega una fila** a `workspace.default.taco_bueno_kpi_snapshot` (append-only) con los KPIs globales del pipeline y el estado del DQ en ese momento. Es la tabla de comparación: al acumular una fila por semana, se puede ver la evolución del pipeline y detectar drift en los CSVs fuente (si un KPI salta entre corridas, cambió algo).

* Escribir aquí es **idempotente por corrida** (un INSERT por ejecución, sin CTAS: nunca borra historia).
* Para comparar: `SELECT * FROM taco_bueno_kpi_snapshot ORDER BY run_ts DESC`, o deltas entre corridas con `LAG() OVER (ORDER BY run_ts)`.

* **Fase 3:** desde la corrección de montos a nivel check, `total_revenue` = revenue bruto a nivel orden, `avg_ticket` = revenue bruto por ORDEN y `total_discount_amount` = descuento a nivel check (contado una vez por check). La columna `metric_convention` distingue las filas nuevas (`check_level`) de las históricas de Fase 2 (convención por línea, inflada ×~6). `dq_fail` ahora cuenta solo status = FAIL (antes contaba WARN como fallo).

In [0]:
# Snapshot append-only: UNA fila por corrida con KPIs globales + estado DQ (para comparar corridas del Job)
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.default.taco_bueno_kpi_snapshot (
  run_ts TIMESTAMP COMMENT 'Momento de la corrida',
  run_source STRING COMMENT 'Origen del snapshot (notebook orquestador)',
  dq_total INT COMMENT 'Total de checks DQ ejecutados',
  dq_pass INT COMMENT 'Checks DQ en PASS',
  dq_fail INT COMMENT 'Checks DQ en FAIL',
  silver_rows BIGINT COMMENT 'Filas en taco_bueno_silver',
  months_covered INT COMMENT 'Meses distintos en gold_monthly',
  total_transactions BIGINT COMMENT 'Transacciones totales',
  total_orders BIGINT COMMENT 'Órdenes totales',
  total_revenue DOUBLE COMMENT 'Revenue neto total ($)',
  avg_ticket DOUBLE COMMENT 'Ticket promedio = revenue / transacciones',
  total_discount_amount DOUBLE COMMENT 'Descuento total otorgado ($)',
  discount_as_pct_of_revenue DECIMAL(10,2) COMMENT 'Descuento como % del revenue',
  trigger_cadence STRING COMMENT 'Cadencia del Job que disparó la corrida: semanal/quincenal/mensual; manual si fue interactiva'
) USING DELTA
""")

# Fase 3: convención de métricas de esta fila. 'check_level' = montos a nivel orden
# contados UNA vez por check; NULL = filas históricas con la convención por línea de
# Fase 2 (revenue inflado ×~6). También se corrige el comentario de avg_ticket.
spark.sql("""ALTER TABLE workspace.default.taco_bueno_kpi_snapshot ADD COLUMN IF NOT EXISTS metric_convention STRING COMMENT 'check_level = montos a nivel orden (Fase 3); NULL = convención por línea (Fase 2)'""")
spark.sql("""ALTER TABLE workspace.default.taco_bueno_kpi_snapshot ALTER COLUMN avg_ticket COMMENT 'Revenue bruto por orden (check_level) o promedio por línea (Fase 2)'""")

# Cadencia del Job: cada Job (semanal/quincenal/mensual) pasa este parámetro; por defecto 'manual' si se corre interactivo
dbutils.widgets.text("pipeline_cadence", "manual")
job_cadence = dbutils.widgets.get("pipeline_cadence")
print(f"Cadencia de la corrida: {job_cadence}")

spark.sql(f"""
INSERT INTO workspace.default.taco_bueno_kpi_snapshot
SELECT
  current_timestamp(),
  '05_Pipeline_Job',
  (SELECT COUNT(*) FROM workspace.default.taco_bueno_dq_results),
  (SELECT COUNT(*) FROM workspace.default.taco_bueno_dq_results WHERE status = 'PASS'),
  (SELECT COUNT(*) FROM workspace.default.taco_bueno_dq_results WHERE status = 'FAIL'),
  (SELECT COUNT(*) FROM workspace.default.taco_bueno_silver),
  m.months, m.transactions, m.orders, m.revenue_gross,
  ROUND(m.revenue_gross / m.orders, 2),
  m.discount_amount_check,
  ROUND(m.discount_amount_check / m.revenue_gross * 100, 2),
  '{job_cadence}',
  'check_level'
FROM (
  SELECT COUNT(DISTINCT month) AS months,
         SUM(transactions) AS transactions, SUM(orders) AS orders,
         SUM(revenue_gross) AS revenue_gross, SUM(discount_amount_check) AS discount_amount_check
  FROM workspace.default.taco_bueno_gold_monthly
) m
""")

print("Snapshot registrado. Historial acumulado:")
display(spark.sql("SELECT run_ts, trigger_cadence, dq_total, dq_pass, dq_fail, silver_rows, months_covered, total_transactions, total_orders, total_revenue, avg_ticket, total_discount_amount, discount_as_pct_of_revenue, metric_convention FROM workspace.default.taco_bueno_kpi_snapshot ORDER BY run_ts DESC"))

In [0]:
# Liberar el lock de esta corrida. Si el pipeline falló antes de llegar aquí,
# el TTL de la celda del lock lo expira automáticamente en la siguiente corrida.
spark.sql(f"""
UPDATE workspace.default.taco_bueno_pipeline_lock
SET released_at = current_timestamp()
WHERE lock_token = '{run_token}' AND released_at IS NULL
""")
print("🔓 Lock liberado — pipeline completo.")